# Azure AI Evaluation SDK v2.0 — POC

> Real SDK evaluators decorated with `@metric`, running through the evee engine. Local or cloud via `azure_ai_project`.


In [ ]:
import os, json

if os.path.basename(os.getcwd()) == "demo":
    os.chdir("..")

from azure.ai.evaluation._eval_v2 import evaluate_v2
from azure.ai.evaluation import RelevanceEvaluator, F1ScoreEvaluator

DATA = "demo/data.jsonl"
PROJECT = "https://foundry-evee-ko9z2s7c.cognitiveservices.azure.com/api/projects/foundry-project-evee-ko9z2s7c"

def _show(result):
    """Display engine results cleanly."""
    from pathlib import Path
    import json as _json
    print(f"Status: {result['status']}  Records: {result['total_records']}")
    output_dir = Path(result["output_path"])
    summaries = sorted(output_dir.glob("*_summary.json"))
    if len(summaries) > 1:
        print(f"\n{'Variant':<55} {'Metrics'}")
        print("-" * 80)
    for sf in summaries:
        s = _json.load(open(sf))
        variant = s.get("model", sf.stem.replace("_summary", ""))
        # Shorten variant name for readability
        if "__" in variant:
            parts = variant.split("__", 1)[1] if "__" in variant else variant
            variant = parts
        metrics_parts = []
        for mn, md in s.get("aggregated_metrics", {}).items():
            for k, v in md.items():
                metrics_parts.append(f"{k}={v}")
        if len(summaries) > 1:
            print(f"  {variant:<53} {', '.join(metrics_parts)}")
        else:
            for part in metrics_parts:
                print(f"  {part}")
    # Show first record detail (from first variant only)
    results_files = sorted(output_dir.glob("*_results.jsonl"))
    if results_files:
        first = _json.loads(open(results_files[0]).readline())
        for mn, scores in first.get("metrics", {}).items():
            score = scores.get(mn, "?")
            model = scores.get(f"{mn}_model", "")
            reason = scores.get(f"{mn}_reason", "")
            parts = [f"  {mn}: {score}"]
            if model: parts.append(f"model={model}")
            print("  ".join(parts))
            if reason: print(f"    reason: {reason[:100]}")
            break  # just first metric of first record
    import shutil; shutil.rmtree(output_dir.parent.parent, ignore_errors=True)
print("Ready")


## How It Works

The real SDK evaluators (`RelevanceEvaluator`, `F1ScoreEvaluator`) are decorated with `@metric` at the bottom of their source files. This registers them in the evee engine's metric registry — making them discoverable from YAML config and runnable through the engine's parallel execution pipeline.

```python
# Added to _evaluators/_f1_score/_f1_score.py:
@_evee_metric(name='f1_score')
class _F1ScoreEveeMetric(_EveeBaseMetric):
    def __init__(self, connections_registry=None, context=None, **kwargs):
        self._evaluator = F1ScoreEvaluator()

    def compute(self, response='', ground_truth='', **kwargs):
        return self._evaluator(response=response, ground_truth=ground_truth)

    def aggregate(self, scores):
        vals = [s.get('f1_score', 0) for s in scores]
        return {'f1_score_mean': sum(vals) / len(vals)}
```

When the engine encounters `name: "relevance"` in the YAML config, it looks up the registry, finds the decorated evaluator, and if a cloud connection is configured, the evaluator automatically uses the Foundry model.


## 1. Local Evaluation — No Model Needed

F1 is a deterministic NLP metric from the real SDK. Runs instantly, offline, no API key.


In [ ]:
result_local = evaluate_v2(
    data=DATA,
    evaluators={"f1": F1ScoreEvaluator},
    evaluator_config={"f1": {"column_mapping": {
        "response": "${data.answer}",
        "ground_truth": "${data.context}",
    }}},
)
_show(result_local)


## 2. Cloud Evaluation — via `azure_ai_project`

Same call — add `azure_ai_project`. The function connects to the Foundry project via `AIProjectClient`, discovers the deployed model (GPT-4.1-mini), and injects the connection into the engine. The `@metric`-decorated `RelevanceEvaluator` picks up the connection and calls the real LLM-as-judge.


In [ ]:
result_cloud = evaluate_v2(
    data=DATA,
    evaluators={"relevance": RelevanceEvaluator},
    evaluator_config={"relevance": {"column_mapping": {
        "query": "${data.question}",
        "response": "${data.answer}",
    }}},
    azure_ai_project=PROJECT,
)
_show(result_cloud)


## 3. Local + Cloud Together

F1 (local, instant) + Relevance (cloud, GPT-4.1-mini judge) — one call, both through the engine.


In [ ]:
result_both = evaluate_v2(
    data=DATA,
    evaluators={"relevance": RelevanceEvaluator, "f1": F1ScoreEvaluator},
    evaluator_config={
        "relevance": {"column_mapping": {"query": "${data.question}", "response": "${data.answer}"}},
        "f1": {"column_mapping": {"response": "${data.answer}", "ground_truth": "${data.context}"}},
    },
    azure_ai_project=PROJECT,
)
_show(result_both)


## 4. Config-Driven via YAML

No Python needed. The YAML references the registered metric names, the engine creates the evaluators, and connections provide the model endpoint.


In [ ]:
with open("demo/evals.yaml") as f:
    print(f.read())


In [ ]:
result_config = evaluate_v2(config="demo/evals.yaml")
_show(result_config)


### Output Files

The engine persists per-record JSONL + summary JSON — structured, reproducible, uploadable to Foundry.


In [ ]:
# Re-run to show output files (previous _show cleaned up)
result_files = evaluate_v2(config="demo/evals.yaml")
from pathlib import Path
output_dir = Path(result_files["output_path"])

print("Output files:")
for f in sorted(output_dir.iterdir()):
    print(f"  {f.name:<45} {f.stat().st_size:>6} bytes")

import shutil; shutil.rmtree(output_dir.parent.parent, ignore_errors=True)


## 5. CLI


In [ ]:
import subprocess

proc = subprocess.run(
    ["python", "-m", "azure.ai.evaluation.cli", "run", "-c", "demo/evals.yaml"],
    capture_output=True, text=True,
)
print(proc.stdout)


## 6. Prompt Strategy Comparison — Real API Calls

A common evaluation task: **which prompt strategy produces better answers?**

We define a `@model` class that calls GPT-4.1-mini with three different system prompts:

| Strategy | System Prompt | What It Tests |
|----------|--------------|---------------|
| **baseline** | "Answer concisely" | Raw model quality with no guidance |
| **single_shot** | 1 example Q&A, then the question | Does one example improve quality? |
| **few_shot** | 3 example Q&As, then the question | Do more examples help further? |

The engine auto-discovers the `@model(name="prompt_compare")` from `demo/prompt_models.py`, expands the parameter grid (3 strategies × 10 questions = 30 API calls), and scores each answer with RelevanceEvaluator + F1.


In [ ]:
# Show the @model class
with open("demo/prompt_models.py") as f:
    print(f.read())


In [ ]:
# Show the comparison config
with open("demo/prompt_comparison.yaml") as f:
    print(f.read())


In [ ]:
# Engine auto-discovers @model from demo/prompt_models.py — no import needed
result_compare = evaluate_v2(config="demo/prompt_comparison.yaml")

print(f"Variants: {result_compare['models_evaluated']}  Records: {result_compare['total_records']}")
_show(result_compare)


## Summary

| Mode | Code | What Happens |
|------|------|-------------|
| **Local** | `evaluate_v2(data=..., evaluators={"f1": F1ScoreEvaluator})` | Engine runs `@metric`-decorated evaluator locally |
| **Cloud** | `evaluate_v2(..., azure_ai_project=PROJECT)` | Connects via `AIProjectClient`, injects model, engine runs `@metric` with cloud model |
| **Config** | `evaluate_v2(config="evals.yaml")` | YAML defines metrics + connections, engine does everything |
| **CLI** | `python -m azure.ai.evaluation.cli run -c evals.yaml` | Same engine, from terminal |

**No bypass. No bridge. Real SDK evaluators decorated with `@metric`, running through the evee engine.**
